# Recommendation UI — Enter a User, See What They'd Like

This notebook is an **interactive companion** to `content-based.ipynb`. Instead of querying recommendations one `product_id` at a time, you enter (or pick) a **`user_id`**, and the UI shows two things side by side:

- **Left: Purchase History** — the products that user has actually bought, as *proof* of what they're into.
- **Right: Recommended Products** — new products picked by the content-based recommender, with a note on *which past purchase* triggered each suggestion.

Run all cells top to bottom, then use the input box + button at the bottom to try any user.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

## 2. Rebuild the Recommendation Engine

This mirrors the pipeline from `content-based.ipynb`: load the raw event log, collapse it into a one-row-per-product catalog, engineer a text "content" profile (category hierarchy + brand + price tier), vectorize with TF-IDF, and fit a Nearest Neighbors model on cosine distance. If you already have these objects (`products`, `tfidf_matrix`, `nn_model`) from that notebook in the same kernel, this just recreates them fresh so this notebook can run standalone.

In [2]:
df = pd.read_csv('kz.csv')
df = df.drop_duplicates()  # the raw log contains exact duplicate line-item rows

products = (
    df.groupby('product_id')
      .agg(
          category_id=('category_id', 'first'),
          category_code=('category_code', 'first'),
          brand=('brand', lambda s: s.mode().iat[0] if not s.mode().empty else 'unknown'),
          price=('price', 'mean'),
      )
      .reset_index()
)
products['brand'] = products['brand'].fillna('unknown').str.lower().str.strip()
products['price'] = products['price'].round(2)

def split_category(code):
    parts = str(code).split('.')
    parts += [parts[-1]] * (3 - len(parts))
    return parts[0], parts[1], parts[2]

products[['category_l1', 'category_l2', 'category_l3']] = products['category_code'].apply(
    lambda c: pd.Series(split_category(c))
)
products['price_tier'] = pd.qcut(products['price'], q=4, labels=['budget', 'mid', 'premium', 'luxury'], duplicates='drop')

products['content'] = (
    products['category_l1'] + ' ' + products['category_l2'] + ' ' + products['category_l3'] + ' ' +
    (products['brand'] + ' ') * 2 + products['price_tier'].astype(str)
)

tfidf = TfidfVectorizer(token_pattern=r"[a-zA-Z0-9]+")
tfidf_matrix = tfidf.fit_transform(products['content'])

nn_model = NearestNeighbors(metric='cosine', algorithm='brute')
nn_model.fit(tfidf_matrix)

product_index = pd.Series(products.index, index=products['product_id'])

print(f"Catalog ready: {products.shape[0]:,} products, {df['user_id'].nunique():,} users.")

Catalog ready: 12,325 products, 206,679 users.


## 3. Purchase History Lookup

Given a `user_id`, this pulls every distinct product they've purchased from the event log — sorted most-recent-first — and joins in the product's catalog details (category, brand, price). This is the "proof" panel: it's exactly what the recommendations are based on.

In [3]:
def get_purchase_history(user_id):
    user_events = df[df['user_id'] == user_id].copy()
    if user_events.empty:
        return pd.DataFrame(columns=['event_time', 'product_id', 'category_code', 'brand', 'price', 'order_id'])

    user_events = user_events.sort_values('event_time', ascending=False)
    user_events = user_events.drop_duplicates(subset=['product_id', 'order_id'])
    return user_events[['event_time', 'product_id', 'category_code', 'brand', 'price', 'order_id']].reset_index(drop=True)

## 4. User-Level Recommendation Function

A user doesn't have a single "content" vector — they have a whole purchase history. So `recommend_for_user`:

1. Looks up every product the user has bought.
2. For **each** purchased product, finds its nearest neighbors (candidate recommendations) via the same content-based model used in the product-to-product notebook.
3. Merges all candidates together, keeping each candidate's **best (highest) similarity score** and remembering *which purchased product* produced that best match — this gives an explainable "similar to your earlier purchase of X" reason.
4. Removes anything the user has already bought (no point recommending what they own).
5. Returns the top-N candidates ranked by similarity.

In [4]:
def recommend_for_user(user_id, top_n=10, neighbors_per_product=15):
    history = get_purchase_history(user_id)
    if history.empty:
        return history, pd.DataFrame(columns=['product_id', 'category_code', 'brand', 'price', 'price_tier', 'similarity', 'similar_to'])

    purchased_ids = history['product_id'].unique()
    purchased_ids_in_catalog = [pid for pid in purchased_ids if pid in product_index]

    candidates = {}  # product_id -> (best_similarity, source_product_id)
    for pid in purchased_ids_in_catalog:
        idx = product_index[pid]
        distances, indices = nn_model.kneighbors(tfidf_matrix[idx], n_neighbors=neighbors_per_product)
        for dist, cand_idx in zip(distances[0], indices[0]):
            cand_id = products.iloc[cand_idx]['product_id']
            if cand_id in purchased_ids:  # already owned, skip
                continue
            sim = 1 - dist
            if cand_id not in candidates or sim > candidates[cand_id][0]:
                candidates[cand_id] = (sim, pid)

    if not candidates:
        return history, pd.DataFrame(columns=['product_id', 'category_code', 'brand', 'price', 'price_tier', 'similarity', 'similar_to'])

    rec_ids = list(candidates.keys())
    recs = products[products['product_id'].isin(rec_ids)].copy()
    recs['similarity'] = recs['product_id'].map(lambda pid: candidates[pid][0])
    recs['similar_to'] = recs['product_id'].map(lambda pid: candidates[pid][1])

    recs = recs.sort_values('similarity', ascending=False).head(top_n)
    return history, recs[['product_id', 'category_code', 'brand', 'price', 'price_tier', 'similarity', 'similar_to']].reset_index(drop=True)

## 5. Quick Non-Interactive Test

Before wiring up the UI widgets, let's sanity-check the function directly on a real, active user (someone with more than one purchase, so the recommendations have something to work with).

In [5]:
sample_user_id = df['user_id'].value_counts().index[0]  # most active user in the dataset
history, recs = recommend_for_user(sample_user_id, top_n=5)

print(f"User: {sample_user_id}  |  {len(history)} purchase(s) on record")
print("\nPurchase history:")
display(history)
print("\nRecommendations:")
display(recs)

User: 1515915625505835892  |  184 purchase(s) on record

Purchase history:


,event_time,product_id,category_code,brand,price,order_id
0,2020-11-19 11:51:22 UTC,1515966223509298406,construction.tools.screw,samsung,32.15,2388440981134687953
1,2020-11-06 22:38:28 UTC,1515966223509123319,computers.peripherals.keyboard,steelseries,208.31,2388440981134667826
2,2020-10-25 08:15:00 UTC,1515966223509131621,furniture.kitchen.table,oral-b,39.33,2388440981134634072
3,2020-10-23 05:59:35 UTC,2273948277256749633,computers.components.cooler,thermaltake,53.68,2388440981134632177
4,2020-10-23 05:59:35 UTC,1515966223509298309,computers.components.memory,kingston,118.73,2388440981134632177
...,...,...,...,...,...,...
179,2020-06-26 05:28:53 UTC,1515966223509299907,computers.peripherals.mouse,asus,52.06,2354524289522205305
180,2020-06-25 18:31:47 UTC,1515966223509127463,electronics.smartphone,apple,1433.54,2339497930613850757
181,2020-06-25 10:47:04 UTC,2273948277189640755,computers.peripherals.keyboard,apple,173.13,2354520401578557759
182,2020-06-25 10:16:47 UTC,1515966223518661752,computers.peripherals.mouse,logitech,138.87,2354522255410594060



Recommendations:


,product_id,category_code,brand,price,price_tier,similarity,similar_to
0,1515966223509089256,electronics.tablet,samsung,532.38,luxury,1.0,1515966223546543596
1,1515966223510421575,computers.components.power_supply,gamemax,33.54,mid,1.0,1515966223509298191
2,1515966223517457341,computers.notebook,apple,1951.37,luxury,1.0,1515966223509089372
3,1515966223513916249,computers.peripherals.keyboard,logitech,237.25,premium,1.0,1515966223509130631
4,1515966223512562046,electronics.tablet,samsung,416.64,luxury,1.0,1515966223546543596


## 6. Helper: Render Purchase History + Recommendations Side by Side

This formats both tables as HTML and lays them out in a two-column flex container, so the purchase history (proof) sits right next to the recommendations it produced. The "Similar to" column in the recommendations shows which purchased product triggered each suggestion.

In [6]:
def render_side_by_side(user_id, history, recs):
    if history.empty:
        return HTML(f"<p style='color:#b00020;font-family:sans-serif;'>No purchase history found for user_id <b>{user_id}</b>. Try another ID.</p>")

    def short_id(pid, length=9):
        """Show a shortened product id with the full id available on hover (tooltip)."""
        s = str(pid)
        short = s if len(s) <= length else s[:length] + "\u2026"
        return f'<span title="{s}">{short}</span>'

    hist_display = history.copy()
    hist_display['price'] = hist_display['price'].map(lambda x: f"${x:,.2f}")
    hist_display['product_id'] = hist_display['product_id'].map(short_id)

    rec_display = recs.copy()
    if not rec_display.empty:
        rec_display['price'] = rec_display['price'].map(lambda x: f"${x:,.2f}")
        rec_display['similarity'] = rec_display['similarity'].map(lambda x: f"{x:.0%}")
        rec_display['similar_to'] = rec_display['similar_to'].map(short_id)
        rec_display['product_id'] = rec_display['product_id'].map(short_id)
        rec_display = rec_display.rename(columns={
            'product_id': 'Product ID', 'category_code': 'Category', 'brand': 'Brand',
            'price': 'Price', 'price_tier': 'Tier', 'similarity': 'Match', 'similar_to': 'Similar to (purchase)'
        })

    hist_display = hist_display.rename(columns={
        'event_time': 'Purchased At', 'product_id': 'Product ID', 'category_code': 'Category',
        'brand': 'Brand', 'price': 'Price', 'order_id': 'Order ID'
    })

    style = """
    <style>
        .rec-ui-wrap { display:flex; gap:20px; font-family:-apple-system,Segoe UI,Roboto,sans-serif;
                       align-items:flex-start; flex-wrap:wrap; }
        .rec-ui-col { flex:1 1 460px; min-width:0; max-width:100%; }
        .rec-ui-col h4 { margin:0 0 8px 0; white-space:nowrap; }
        .rec-ui-scroll { overflow-x:auto; border:1px solid #e3e3e8; border-radius:6px; }
        .rec-ui-col table { border-collapse:collapse; width:100%; min-width:520px; font-size:12.5px; table-layout:auto; }
        .rec-ui-col th, .rec-ui-col td { padding:6px 10px; border-bottom:1px solid #eee; white-space:nowrap; }
        .rec-ui-col th { background:#f2f2f5; text-align:left; border-bottom:2px solid #ddd; position:sticky; top:0; }
        .rec-ui-col.history th { background:#eaf3ea; }
        .rec-ui-col.recs th { background:#eaeef7; }
        .rec-ui-col td span[title] { border-bottom:1px dotted #999; cursor:help; }
    </style>
    """

    hist_html = hist_display.to_html(index=False, escape=False)
    recs_html = rec_display.to_html(index=False, escape=False) if not rec_display.empty else "<p>No recommendations available for this user.</p>"

    html = f"""
    {style}
    <div class="rec-ui-wrap">
        <div class="rec-ui-col history">
            <h4>🛒 Purchase History for user <code>{user_id}</code> (proof — {len(history)} item{'s' if len(history)!=1 else ''})</h4>
            <div class="rec-ui-scroll">{hist_html}</div>
        </div>
        <div class="rec-ui-col recs">
            <h4>✨ Recommended For This User</h4>
            <div class="rec-ui-scroll">{recs_html}</div>
        </div>
    </div>
    <p style="font-family:-apple-system,Segoe UI,Roboto,sans-serif;font-size:11px;color:#888;margin-top:6px;">
        Tip: Product IDs are shortened for readability — hover over one to see the full ID. Scroll a table horizontally if it doesn't fit.
    </p>
    """
    return HTML(html)

## 7. Interactive UI

Enter a `user_id` in the box below (or pick one from the dropdown of active sample users), then click **Get Recommendations**. Use **🎲 Random Active User** to instantly try a different shopper.

The purchase history on the left is the *proof* — it's exactly what the model used to generate the suggestions on the right.

In [ ]:
# Sample of active users (>=3 distinct purchases) to make the dropdown useful for demoing
active_users = (
    df.groupby('user_id')['product_id'].nunique()
      .loc[lambda s: s >= 3]
      .sort_values(ascending=False)
      .head(200)
      .index.astype(str).tolist()
)

user_input = widgets.Combobox(
    placeholder='Enter or select a user_id',
    options=active_users,
    description='User ID:',
    ensure_option=False,
    layout=widgets.Layout(width='420px'),
)

get_button = widgets.Button(description='Get Recommendations', button_style='primary', icon='search')
random_button = widgets.Button(description='Random Active User', icon='random')
output_area = widgets.Output()

def on_get_clicked(b):
    with output_area:
        clear_output(wait=True)
        raw_value = user_input.value.strip()
        if not raw_value:
            display(HTML("<p style='color:#b00020;'>Please enter or select a user_id.</p>"))
            return
        try:
            uid = int(raw_value)
        except ValueError:
            display(HTML(f"<p style='color:#b00020;'>'{raw_value}' is not a valid user_id (expected a number).</p>"))
            return
        history, recs = recommend_for_user(uid, top_n=10)
        display(render_side_by_side(uid, history, recs))

def on_random_clicked(b):
    uid = int(np.random.choice(active_users))
    user_input.value = str(uid)
    on_get_clicked(b)

get_button.on_click(on_get_clicked)
random_button.on_click(on_random_clicked)

ui = widgets.VBox([
    widgets.HBox([user_input, get_button, random_button]),
    output_area,
])
display(ui)

### Usage notes

- The **User ID** box accepts either a typed ID or a selection from the dropdown (pre-populated with ~200 active sample users who have 3+ distinct purchases, so results are meaningful).
- **Random Active User** is the fastest way to explore — it fills the box and runs the recommendation in one click.
- If a `user_id` has no purchase history in `kz.csv`, the UI will say so instead of showing an empty table.
- The **"Similar to (purchase)"** column in the recommendations table is the explainability piece: it names the specific product from the user's history that caused each suggestion to surface.